# R&D-7 — внутренний контур: два преобразования сырых эмбеддингов

Самодостаточный ноутбук (pyspark, без импортов `rnd_reports`). Содержит **только две
функции** над таблицей эмбеддингов формата `epk_id × report_dt × emb_{i}_val`. Каждая
принимает один pyspark DataFrame и возвращает его же с **добавленными** колонками:

1. **Скрипт 1 — понижение размерности** (`reduce_embeddings`): StandardScaler + PCA →
   добавляет колонки `red_0, …, red_{red_size-1}`.
2. **Скрипт 2 — вычисление p-score** (`add_propensity_score`): добавляет колонку
   `prop_score` = P(treatment=1), обучив лучший по ROC-AUC из **LogisticRegression /
   GBTClassifier**. Трит — опциональная колонка того же датафрейма: если колонки
   `treatment` нет, она генерируется случайно (`random_state`) и тоже добавляется.
   ROC-AUC здесь — лишь эвристика выбора модели; `prop_score` дальше идёт в
   propensity-score matching (сам матчинг тут не реализуется).

**Во внутреннем контуре:**
- `SparkSession` (`spark`) — **ваш**, со своей конфигурацией; ячейку
  «репозиторный bootstrap Spark» **пропустите**;
- в ячейке конфигурации укажите свою таблицу `INPUT_EMB` и `RED_SIZE`;
- ячейку-генератор синтетики эмбеддингов тоже **пропустите** (она помечена «не запускать»);
- выполните остальные ячейки по порядку.

> ⚠️ Если в вашей таблице **нет** колонки `treatment`, p-score считается на случайно
> сгенерированном трите → он неинформативен (ROC-AUC ≈ 0.5). Реальная ценность p-score —
> когда колонка `treatment` уже присутствует в данных.

In [1]:
# Импорты — только базовый pyspark-стек (без rnd_reports).
import re

from pyspark.sql import functions as F
from pyspark.ml.feature import PCA, StandardScaler, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.classification import GBTClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

## Скрипт 1 — понижение размерности (вшит из `rnd_reports.embeddings`)

Контракт схемы + `reduce_embeddings`. Во внутреннем контуре эти ячейки не правят.

In [2]:
# === Контракт схемы (вшито из rnd_reports.embeddings/contracts.py) ===
# Сырая таблица: epk_id, report_dt, emb_0_val, emb_1_val, ... (легаси-формат col_000 тоже понимается).

EPK_ID, REPORT_DT, TREATMENT = "epk_id", "report_dt", "treatment"
KEY_COLUMNS = (EPK_ID, REPORT_DT)
REDUCED_PREFIX = "red_"          # выход reducer'а: red_0, red_1, ...
PROPENSITY_SCORE = "prop_score"  # выход propensity-скрипта

_EMB_NEW = re.compile(r"^emb_(\d+)_val$")
_EMB_LEGACY = re.compile(r"^col_(\d+)$")


def _embedding_index(name):
    m = _EMB_NEW.match(name) or _EMB_LEGACY.match(name)
    return int(m.group(1)) if m else None


def embedding_feature_columns(df):
    indexed = [(i, c) for c in df.columns if (i := _embedding_index(c)) is not None]
    return [name for _, name in sorted(indexed)]


def reduced_column_names(red_size):
    if red_size < 1:
        raise ValueError(f"red_size должен быть >= 1, получено {red_size}")
    return [f"{REDUCED_PREFIX}{i}" for i in range(red_size)]


def validate_embedding_schema(df):
    missing = [c for c in KEY_COLUMNS if c not in set(df.columns)]
    if missing:
        raise ValueError(f"В эмбеддинг-датасете нет ключевых колонок: {missing}")
    cols = embedding_feature_columns(df)
    if not cols:
        raise ValueError("Не найдено ни одной эмбеддинг-колонки 'emb_{i}_val' (или легаси 'col_{i}')")
    return cols


def reduce_embeddings(df, red_size=5):
    """Скрипт 1: понижение размерности эмбеддингов (StandardScaler + PCA).

    Возвращает ИСХОДНЫЙ df с добавленными колонками red_0..red_{red_size-1}.
    PCA детерминирован (SVD) — сид не нужен.
    """
    feats = validate_embedding_schema(df)
    names = reduced_column_names(red_size)
    assembled = VectorAssembler(inputCols=feats, outputCol="__f__").transform(df)
    scaler = StandardScaler(inputCol="__f__", outputCol="__s__", withMean=True, withStd=True).fit(assembled)
    scaled = scaler.transform(assembled)
    pca = PCA(k=red_size, inputCol="__s__", outputCol="__p__").fit(scaled)
    out = pca.transform(scaled).withColumn("__a__", vector_to_array(F.col("__p__")))
    for i, n in enumerate(names):
        out = out.withColumn(n, F.col("__a__")[i])
    return out.drop("__f__", "__s__", "__p__", "__a__")

## Скрипт 2 — вычисление p-score (вшит из `rnd_reports.embeddings`)

`add_propensity_score`. Трит — опциональная колонка того же датафрейма; при отсутствии
генерируется случайно (`random_state`). Во внутреннем контуре эту ячейку не правят.

In [3]:
# === prop_score (вшито из rnd_reports.embeddings/propensity.py) ===
# Скрипт 2: prop_score = P(treatment=1 | эмбеддинги); из LogReg / GBT берётся лучший по ROC-AUC.

_HASH_MOD = 1000  # гранулярность детерминированного псевдо-трита


def _random_treatment(epk_col, random_state, treatment_share):
    # F.rand нельзя: недетерминирован между материализациями (метка «уехала» бы между fit и
    # оценкой и относительно randomSplit). Хэш epk_id+сид стабилен при любом партиционировании.
    threshold = int(round(treatment_share * _HASH_MOD))
    bucket = F.pmod(F.hash(epk_col, F.lit(random_state)), F.lit(_HASH_MOD))
    return (bucket < F.lit(threshold)).cast("int")


def _prop_candidates(max_iter, reg_param, gbt_max_depth, gbt_max_iter, seed):
    # probability/raw задаём сеттерами: GBTClassifier не принимает их kwargs конструктора.
    est = {
        "logreg": LogisticRegression(featuresCol="__f__", labelCol="__y__",
                                     maxIter=max_iter, regParam=reg_param),
        "gbt": GBTClassifier(featuresCol="__f__", labelCol="__y__",
                             maxDepth=gbt_max_depth, maxIter=gbt_max_iter, seed=seed),
    }
    for e in est.values():
        e.setProbabilityCol("__prob__").setRawPredictionCol("__raw__")
    return est


def add_propensity_score(df, treatment_col=TREATMENT, score_col=PROPENSITY_SCORE,
                         random_state=42, treatment_share=0.5, max_iter=100, reg_param=0.0,
                         gbt_max_depth=5, gbt_max_iter=20, val_fraction=0.3):
    """Скрипт 2: добавляет к df колонку prop_score = P(treatment=1 | эмбеддинги).

    Если колонки treatment нет — генерируется детерминированный псевдо-случайный трит из хэша
    epk_id + random_state и тоже добавляется к выходу. Из LogReg/GBT берётся лучший по ROC-AUC.
    Все исходные колонки сохраняются.
    """
    feats = validate_embedding_schema(df)
    if treatment_col not in df.columns:
        df = df.withColumn(treatment_col, _random_treatment(F.col(EPK_ID), random_state, treatment_share))
    labelled = df.withColumn("__y__", F.col(treatment_col).cast("double"))
    assembled = VectorAssembler(inputCols=feats, outputCol="__f__").transform(labelled)
    train, val = assembled.randomSplit([1 - val_fraction, val_fraction], seed=random_state)
    ev = BinaryClassificationEvaluator(labelCol="__y__", rawPredictionCol="__raw__", metricName="areaUnderROC")
    cands = _prop_candidates(max_iter, reg_param, gbt_max_depth, gbt_max_iter, random_state)
    metrics = {name: float(ev.evaluate(est.fit(train).transform(val))) for name, est in cands.items()}
    best = max(metrics, key=metrics.get)
    print("propensity ROC-AUC:", {k: round(v, 4) for k, v in metrics.items()}, "| выбрана:", best)
    model = _prop_candidates(max_iter, reg_param, gbt_max_depth, gbt_max_iter, random_state)[best].fit(assembled)
    out = model.transform(assembled).withColumn("__pa__", vector_to_array(F.col("__prob__")))
    out = out.withColumn(score_col, F.col("__pa__")[1])
    return out.drop("__f__", "__y__", "__prob__", "__raw__", "__pa__", "prediction")

## Конфигурация

Укажите свои входные таблицы. `SparkSession` (`spark`) во внутреннем контуре — **ваш**.

In [4]:
INPUT_EMB = "data/07_embedding_adjustment_set/emb_raw.parquet"  # epk_id, report_dt, emb_{i}_val [, treatment]
RED_SIZE = 5
SEED = 42

In [5]:
# +++ РЕПОЗИТОРНЫЙ BOOTSTRAP SPARK — НЕ ДЛЯ ВНУТРЕННЕГО КОНТУРА +++
# Во внутреннем контуре `spark` уже создан вашей конфигурацией — ПРОПУСТИТЕ эту ячейку.
# Здесь локальный Spark поднимается только для прогона ноутбука в репозитории.
if "spark" not in globals():
    import os, glob, shutil
    from pyspark.sql import SparkSession

    if not (os.environ.get("JAVA_HOME") and os.path.isdir(os.environ["JAVA_HOME"])):
        cand = sorted(glob.glob("/usr/lib/jvm/*"), reverse=True)
        if (java := shutil.which("java")):
            home = os.path.realpath(java)
            for _ in range(3):  # .../<jdk>/jre/bin/java -> .../<jdk>
                home = os.path.dirname(home)
            cand.insert(0, home)
        for c in cand:
            if os.path.isdir(os.path.join(c, "bin")):
                os.environ["JAVA_HOME"] = c
                break

    spark = (
        SparkSession.builder.master("local[1]").appName("rnd7-internal")
        .config("spark.ui.enabled", "false").config("spark.sql.shuffle.partitions", "4")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("репозиторный Spark", spark.version)

репозиторный Spark 3.4.4


In [6]:
# +++ ГЕНЕРАТОР СИНТЕТИКИ — НЕ ЗАПУСКАТЬ ВО ВНУТРЕННЕМ КОНТУРЕ +++
# Синтез ОДНОЙ демонстрационной таблицы эмбеддингов → parquet (INPUT_EMB).
# Колонку treatment здесь НЕ создаём — скрипт 2 сам сгенерирует случайный трит при её отсутствии.
# Нужен только для проверки запускаемости ноутбука в репозитории.
import numpy as np
import pandas as pd
from pathlib import Path

_rng = np.random.default_rng(SEED)
_n, _k, _n_latent = 4000, 16, 3
_z = _rng.normal(size=(_n, _n_latent))                       # латентные конфаундеры
_W = _rng.normal(size=(_n_latent, _k))
_emb = _z @ _W + _rng.normal(scale=0.5, size=(_n, _k))       # эмбеддинги = латенты + шум
_months = np.array(["2024-01-01", "2024-02-01", "2024-03-01"])
_report_dt = _months[_rng.integers(0, len(_months), size=_n)]

_emb_pd = pd.DataFrame(_emb, columns=[f"emb_{j}_val" for j in range(_k)])
_emb_pd.insert(0, "report_dt", _report_dt)
_emb_pd.insert(0, "epk_id", 1000 + np.arange(_n))

Path(INPUT_EMB).parent.mkdir(parents=True, exist_ok=True)
spark.createDataFrame(_emb_pd).write.mode("overwrite").parquet(INPUT_EMB)
print(f"синтетика записана: {_n} юнитов, {_k} эмбеддинг-колонок")

синтетика записана: 4000 юнитов, 16 эмбеддинг-колонок


In [7]:
# Загрузка таблицы. Во внутреннем контуре INPUT_EMB — ваша реальная таблица эмбеддингов.
# Колонка treatment опциональна: если её нет, скрипт 2 сгенерирует случайный трит сам.
emb = spark.read.parquet(INPUT_EMB)
print("колонки:", emb.columns)
print("есть treatment:", TREATMENT in emb.columns)
display(emb.limit(5).toPandas())

колонки: ['epk_id', 'report_dt', 'emb_0_val', 'emb_1_val', 'emb_2_val', 'emb_3_val', 'emb_4_val', 'emb_5_val', 'emb_6_val', 'emb_7_val', 'emb_8_val', 'emb_9_val', 'emb_10_val', 'emb_11_val', 'emb_12_val', 'emb_13_val', 'emb_14_val', 'emb_15_val']
есть treatment: False


,epk_id,report_dt,emb_0_val,emb_1_val,emb_2_val,emb_3_val,emb_4_val,emb_5_val,emb_6_val,emb_7_val,emb_8_val,emb_9_val,emb_10_val,emb_11_val,emb_12_val,emb_13_val,emb_14_val,emb_15_val
0,1000,2024-01-01,0.386700,2.461619,-2.524029,1.990745,0.964076,1.547253,-4.229097,-0.591109,-0.626682,-0.206244,0.380927,-2.704960,0.684516,-0.877203,-0.824659,-0.477414
1,1001,2024-02-01,-2.105155,3.231162,0.878168,-1.462853,-0.306624,-1.225297,-0.511273,-2.585024,-1.046876,-1.114414,0.689469,-4.569152,4.261411,-1.299537,-3.038901,-1.419793
2,1002,2024-01-01,-0.347417,0.233247,-0.315198,-0.475349,-0.279976,-0.228226,-0.021323,-0.412320,-0.850183,-0.936599,0.213891,-0.824945,0.099039,-0.405309,-0.233105,0.432838
3,1003,2024-02-01,0.574019,-1.055511,-0.130157,1.044985,-0.115584,1.579408,-0.344418,0.398490,0.520068,0.498321,-0.679206,2.520896,-1.024178,2.795302,2.262314,0.565890
4,1004,2024-03-01,0.844067,-1.939272,-0.974482,-0.999793,-0.675547,0.338302,0.152804,1.530257,0.413706,1.421441,-0.042543,3.637684,-2.020274,-0.755419,1.728385,1.142775


## Скрипт 1 — понижение размерности

`reduce_embeddings(emb, red_size=RED_SIZE)` → тот же датафрейм с добавленными `red_0, …`.

In [8]:
reduced = reduce_embeddings(emb, red_size=RED_SIZE)
print("колонки после reduce_embeddings:", reduced.columns)
display(reduced.limit(5).toPandas())

колонки после reduce_embeddings: ['epk_id', 'report_dt', 'emb_0_val', 'emb_1_val', 'emb_2_val', 'emb_3_val', 'emb_4_val', 'emb_5_val', 'emb_6_val', 'emb_7_val', 'emb_8_val', 'emb_9_val', 'emb_10_val', 'emb_11_val', 'emb_12_val', 'emb_13_val', 'emb_14_val', 'emb_15_val', 'red_0', 'red_1', 'red_2', 'red_3', 'red_4']


,epk_id,report_dt,emb_0_val,emb_1_val,emb_2_val,emb_3_val,emb_4_val,emb_5_val,emb_6_val,emb_7_val,emb_8_val,emb_9_val,emb_10_val,emb_11_val,emb_12_val,emb_13_val,emb_14_val,emb_15_val,red_0,red_1,red_2,red_3,red_4
0,1000,2024-01-01,0.386700,2.461619,-2.524029,1.990745,0.964076,1.547253,-4.229097,-0.591109,-0.626682,-0.206244,0.380927,-2.704960,0.684516,-0.877203,-0.824659,-0.477414,0.622190,-2.362363,-0.023584,-1.975121,1.058780
1,1001,2024-02-01,-2.105155,3.231162,0.878168,-1.462853,-0.306624,-1.225297,-0.511273,-2.585024,-1.046876,-1.114414,0.689469,-4.569152,4.261411,-1.299537,-3.038901,-1.419793,3.855902,-1.364080,-3.186711,0.191165,-0.406139
2,1002,2024-01-01,-0.347417,0.233247,-0.315198,-0.475349,-0.279976,-0.228226,-0.021323,-0.412320,-0.850183,-0.936599,0.213891,-0.824945,0.099039,-0.405309,-0.233105,0.432838,0.408155,-0.526132,-0.184053,0.297947,-0.206375
3,1003,2024-02-01,0.574019,-1.055511,-0.130157,1.044985,-0.115584,1.579408,-0.344418,0.398490,0.520068,0.498321,-0.679206,2.520896,-1.024178,2.795302,2.262314,0.565890,-1.111463,0.797809,2.164453,0.284795,0.700500
4,1004,2024-03-01,0.844067,-1.939272,-0.974482,-0.999793,-0.675547,0.338302,0.152804,1.530257,0.413706,1.421441,-0.042543,3.637684,-2.020274,-0.755419,1.728385,1.142775,-2.564040,0.342267,1.439965,1.159731,-0.785371


## Скрипт 2 — вычисление p-score

`add_propensity_score(emb, random_state=SEED)` → тот же датафрейм с добавленными
`treatment` (если её не было) и `prop_score`.

In [9]:
scored = add_propensity_score(emb, random_state=SEED)
print("колонки после add_propensity_score:", scored.columns)
display(scored.select(EPK_ID, REPORT_DT, TREATMENT, PROPENSITY_SCORE).limit(5).toPandas())

propensity ROC-AUC: {'logreg': 0.5205, 'gbt': 0.5073} | выбрана: logreg
колонки после add_propensity_score: ['epk_id', 'report_dt', 'emb_0_val', 'emb_1_val', 'emb_2_val', 'emb_3_val', 'emb_4_val', 'emb_5_val', 'emb_6_val', 'emb_7_val', 'emb_8_val', 'emb_9_val', 'emb_10_val', 'emb_11_val', 'emb_12_val', 'emb_13_val', 'emb_14_val', 'emb_15_val', 'treatment', 'prop_score']


,epk_id,report_dt,treatment,prop_score
0,1000,2024-01-01,1,0.517414
1,1001,2024-02-01,1,0.548094
2,1002,2024-01-01,1,0.512695
3,1003,2024-02-01,1,0.458817
4,1004,2024-03-01,1,0.469157
